# Análise Comparativa Inicial: LLDA vs. LDA Clássico no Dataset AAPD

**Experimento de Origem:** `experimento_comparativo_aapd.py`  
**Dataset:** ArXiv Academic Paper Dataset (AAPD) — 2.000 resumos de artigos científicos com rótulos multi-label.

---

### Objetivos deste Notebook:
1. **Explorar e validar** os dados gerados pelo experimento comparativo (`experimento_aapd_comparativo.csv`).
2. **Comparar a Diversidade Lexical (Distinct-1 e Distinct-2)** e riqueza de vocabulário entre o modelo generativo baseado em LLM (**LLDA**) e a abordagem probabilística tradicional (**LDA Clássico** com $K=54$).
3. **Avaliar a estrutura textual dos tópicos** (comprimento médio em palavras e concisão).
4. **Analisar as métricas de confiança/probabilidade** associadas às inferências de cada técnica.
5. **Examinar a aderência qualitativa** frente às categorias reais do arXiv (`classes_reais_aapd`).


## 1. Configuração do Ambiente e Carregamento dos Dados

Importamos as bibliotecas analíticas e configuramos o `plotly` para gerar visualizações ricas e interativas.


In [1]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Configuração de exibição do Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# Caminho do arquivo de resultados comparativos
csv_path = "experimento_aapd_comparativo.csv"
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.path.dirname(__file__), "experimento_aapd_comparativo.csv")

df = pd.read_csv(csv_path)

print(f"Dataset carregado com sucesso!")
print(f"Total de registros: {len(df):,} linhas | Total de colunas: {len(df.columns)}")
df.info()


Dataset carregado com sucesso!
Total de registros: 2,000 linhas | Total de colunas: 7
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   doc_id                      2000 non-null   int64  
 1   classes_reais_aapd          2000 non-null   str    
 2   topico_llda                 2000 non-null   str    
 3   confianca_llda              2000 non-null   float64
 4   topico_lda_classico         2000 non-null   str    
 5   probabilidade_lda_classico  2000 non-null   float64
 6   resumo_texto                2000 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 109.5 KB


### Verificação Inicial dos Dados

Inspecionamos as 5 primeiras amostras do arquivo consolidado para entender as colunas disponíveis.


In [2]:
# Verificação de valores nulos e prévia dos primeiros registros
print("Contagem de valores ausentes por coluna:")
print(df.isnull().sum())
print("-" * 60)
df.head(5)


Contagem de valores ausentes por coluna:
doc_id                        0
classes_reais_aapd            0
topico_llda                   0
confianca_llda                0
topico_lda_classico           0
probabilidade_lda_classico    0
resumo_texto                  0
dtype: int64
------------------------------------------------------------


,doc_id,classes_reais_aapd,topico_llda,confianca_llda,topico_lda_classico,probabilidade_lda_classico,resumo_texto
0,51,"math.CO, cs.DM",Graph Edge Rabbit Hunting Problem,0.333333,"graph, graphs, time, algorithm, log, problem",0.688232,"we investigate hunters rabbit game , where a set of hunters tries to catch an invisible rabbit that slides along the..."
1,178,"physics.soc-ph, cs.DL",Scientific Research Collaboration Dynamics,0.333333,"problem, algorithm, algorithms, optimization, power, method",0.676931,collaborations and citations within scientific research grow simultaneously and interact dynamically modelling the c...
2,209,"cs.SI, physics.soc-ph",network seed identification measures,0.333333,"systems, algorithm, based, methods, paper, flow",0.669995,recently an increasing amount of research is devoted to the question of how the most influential nodes \( seeds \) c...
3,228,"cs.IT, math.IT",Quantum code construction from cyclic codes,0.333333,"codes, code, decoding, complexity, linear, coding",0.405525,"let r f 2 u f 2 u 2 f 2 be a non chain finite commutative ring , where u 3 u in this paper , we mainly study the con..."
4,285,"cs.DC, cs.DS",Sparse Matrix Storage Format Adaptation,0.333333,"cyclic, set, algorithm, adaptive, results, using",0.976061,we present new adaptive format for storing sparse matrices on gpu we compare it with several other formats including...


## 2. Análise de Diversidade Lexical e Riqueza Vocabular

A diversidade dos tópicos gerados é quantificada através das seguintes métricas:
- **Distinct-1 (Unigramas):** Razão entre palavras únicas e o total de palavras geradas em todos os tópicos.
- **Distinct-2 (Bigramas):** Razão entre pares de palavras consecutivos únicos e o total de bigramas gerados.
- **Tópicos Únicos:** Quantidade de rótulos de tópicos distintos atribuídos entre os 2.000 documentos.
- **Vocabulário Total Único:** Tamanho do léxico empregado na caracterização dos tópicos.


In [3]:
def calc_distinct_n(topics, n=2):
    """Calcula a métrica Distinct-N (diversidade de n-gramas)."""
    ngrams = []
    for t in topics:
        words = str(t).lower().replace(',', '').split()
        if len(words) < n:
            continue
        ngrams.extend([tuple(words[i:i+n]) for i in range(len(words)-n+1)])
    return (len(set(ngrams)) / len(ngrams) * 100) if ngrams else 0.0

def calc_unique_vocab(topics):
    """Retorna o tamanho do vocabulário único."""
    vocab = set()
    for t in topics:
        words = str(t).lower().replace(',', '').split()
        vocab.update(words)
    return len(vocab)

# Cálculo para LLDA
d1_llda = calc_distinct_n(df['topico_llda'], n=1)
d2_llda = calc_distinct_n(df['topico_llda'], n=2)
vocab_llda = calc_unique_vocab(df['topico_llda'])
uniq_llda = df['topico_llda'].nunique()

# Cálculo para LDA Clássico
d1_lda = calc_distinct_n(df['topico_lda_classico'], n=1)
d2_lda = calc_distinct_n(df['topico_lda_classico'], n=2)
vocab_lda = calc_unique_vocab(df['topico_lda_classico'])
uniq_lda = df['topico_lda_classico'].nunique()

df_diversity = pd.DataFrame({
    'Métrica': [
        'Distinct-1 (Unigramas)',
        'Distinct-2 (Bigramas)',
        'Tópicos Únicos (em 2.000 docs)',
        'Vocabulário Único (Palavras)'
    ],
    'LLDA (LLM Generativo)': [f"{d1_llda:.2f}%", f"{d2_llda:.2f}%", f"{uniq_llda:,}", f"{vocab_llda:,}"],
    'LDA Clássico (K=54)': [f"{d1_lda:.2f}%", f"{d2_lda:.2f}%", f"{uniq_lda:,}", f"{vocab_lda:,}"]
})

display(df_diversity)


,Métrica,LLDA (LLM Generativo),LDA Clássico (K=54)
0,Distinct-1 (Unigramas),27.20%,1.42%
1,Distinct-2 (Bigramas),84.01%,2.63%
2,Tópicos Únicos (em 2.000 docs),"1,989",54
3,Vocabulário Único (Palavras),"2,322",170


### Visualização 1: Comparação de Distinct-1 e Distinct-2

Gráfico interativo ilustrando a diferença expressiva na variedade lexical entre as duas abordagens.


In [4]:
fig_div = go.Figure(data=[
    go.Bar(
        name='LLDA (LLM Generativo)',
        x=['Distinct-1 (Unigramas)', 'Distinct-2 (Bigramas)'],
        y=[d1_llda, d2_llda],
        text=[f"{d1_llda:.1f}%", f"{d2_llda:.1f}%"],
        textposition='auto',
        marker_color='#1f77b4'
    ),
    go.Bar(
        name='LDA Clássico (K=54)',
        x=['Distinct-1 (Unigramas)', 'Distinct-2 (Bigramas)'],
        y=[d1_lda, d2_lda],
        text=[f"{d1_lda:.1f}%", f"{d2_lda:.1f}%"],
        textposition='auto',
        marker_color='#ff7f0e'
    )
])

fig_div.update_layout(
    title='Comparação de Diversidade Lexical: Distinct-1 vs Distinct-2',
    yaxis_title='Diversidade (%)',
    barmode='group',
    template='plotly_white',
    height=480,
    legend=dict(x=0.02, y=0.98)
)

fig_div.show()


## 3. Comprimento e Estrutura Textual dos Tópicos

Analisamos o número de palavras empregadas em cada rótulo de tópico.
- **LLDA:** Instruído via prompt a sintetizar o tema em 2 a 5 palavras.
- **LDA Clássico:** Configurado com as top 6 palavras mais prováveis de cada distribuição latente.


In [5]:
df['palavras_llda'] = df['topico_llda'].apply(lambda x: len(str(x).split()))
df['palavras_lda'] = df['topico_lda_classico'].apply(lambda x: len(str(x).replace(',', '').split()))

print("Estatísticas descritivas do número de palavras por tópico:")
stats_words = pd.DataFrame({
    'LLDA (Palavras/Tópico)': df['palavras_llda'].describe(),
    'LDA (Palavras/Tópico)': df['palavras_lda'].describe()
})
display(stats_words.round(2))


Estatísticas descritivas do número de palavras por tópico:


,LLDA (Palavras/Tópico),LDA (Palavras/Tópico)
count,2000.00,2000.0
mean,4.27,6.0
std,1.91,0.0
min,2.00,6.0
25%,3.00,6.0
50%,4.00,6.0
75%,5.00,6.0
max,19.00,6.0


### Visualização 2: Distribuição do Comprimento dos Tópicos


In [ ]:
fig_len = go.Figure()

fig_len.add_trace(go.Histogram(
    x=df['palavras_llda'],
    name='LLDA (LLM)',
    marker_color='#1f77b4',
    opacity=0.75,
    xbins=dict(start=1, end=20, size=1)
))

fig_len.add_trace(go.Histogram(
    x=df['palavras_lda'],
    name='LDA Clássico (Fixo em 6)',
    marker_color='#ff7f0e',
    opacity=0.75,
    xbins=dict(start=1, end=20, size=1)
))

fig_len.update_layout(
    title='Distribuição da Quantidade de Palavras por Rótulo de Tópico',
    xaxis_title='Número de Palavras no Tópico',
    yaxis_title='Frequência de Documentos',
    barmode='overlay',
    template='plotly_white',
    height=480,
    legend=dict(x=0.75, y=0.95)
)

fig_len.show()


## 4. Análise das Métricas de Confiança e Probabilidade

As duas técnicas utilizam métricas de confiança de naturezas distintas:
1. **`confianca_llda`:** Proporção de votos do tópico predominante em múltiplas amostragens com temperatura $0.7$ (acordo inter-amostras).
2. **`probabilidade_lda_classico`:** Probabilidade máxima a posteriori da distribuição latente documento-tópico ($P(z|d)$).


In [6]:
print("Resumo Estatístico de Confiança e Probabilidade:")
stats_conf = df[['confianca_llda', 'probabilidade_lda_classico']].describe()
display(stats_conf.round(3))


Resumo Estatístico de Confiança e Probabilidade:


,confianca_llda,probabilidade_lda_classico
count,2000.000,2000.000
mean,0.545,0.639
std,0.146,0.249
min,0.333,0.180
25%,0.500,0.421
50%,0.500,0.603
75%,0.500,0.905
max,1.000,0.994


### Visualização 3: Distribuição de Confiança (LLDA) vs. Probabilidade (LDA)


In [ ]:
fig_box = go.Figure()

fig_box.add_trace(go.Box(
    y=df['confianca_llda'],
    name='Confiança LLDA (Acordo de Amostragens)',
    marker_color='#1f77b4',
    boxpoints='outliers'
))

fig_box.add_trace(go.Box(
    y=df['probabilidade_lda_classico'],
    name='Probabilidade Posterior LDA',
    marker_color='#ff7f0e',
    boxpoints='outliers'
))

fig_box.update_layout(
    title='Comparação de Incerteza: Confiança LLDA vs. Probabilidade LDA',
    yaxis_title='Valor da Métrica (0.0 a 1.0)',
    template='plotly_white',
    height=500
)

fig_box.show()


## 5. Frequência dos Tópicos e Concentração no Corpus

Verificamos a concentração de tópicos nos 2.000 documentos:
- Como o LDA possui $K=54$ tópicos fixos, certos tópicos agrupam dezenas de artigos.
- Já no LLDA, a geração aberta resulta em tópicos altamente específicos para cada manuscrito.


In [7]:
# Top 10 tópicos mais atribuídos pelo LDA Clássico
top_lda = df['topico_lda_classico'].value_counts().head(10).reset_index()
top_lda.columns = ['Palavras-Chave do Tópico LDA', 'Documentos Atribuídos']

display(top_lda)


,Palavras-Chave do Tópico LDA,Documentos Atribuídos
0,"channel, network, relay, power, transmission, multiple",129
1,"graph, graphs, time, algorithm, log, problem",100
2,"problem, algorithm, functions, problems, algorithms, function",90
3,"codes, code, decoding, complexity, linear, coding",75
4,"channel, rate, capacity, interference, region, achievable",60
5,"data, based, learning, method, methods, proposed",57
6,"network, networks, data, nodes, algorithm, high",56
7,"algorithm, based, user, networks, time, paper",53
8,"sensing, signal, sparse, signals, performance, compressed",53
9,"based, method, data, error, paper, key",48


### Visualização 4: Top 10 Tópicos Mais Atribuídos pelo LDA Clássico


In [ ]:
fig_top_lda = px.bar(
    top_lda,
    x='Documentos Atribuídos',
    y='Palavras-Chave do Tópico LDA',
    orientation='h',
    title='Top 10 Tópicos com Maior Frequência no Corpus (LDA Clássico)',
    text='Documentos Atribuídos',
    color='Documentos Atribuídos',
    color_continuous_scale='Oranges'
)

fig_top_lda.update_layout(
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=500
)

fig_top_lda.show()


## 6. Mapeamento com as Categorias Oficiais do arXiv (Ground Truth)

O dataset AAPD é composto por anotações multi-label de artigos científicos em áreas de Ciência da Computação, Matemática e Física.  
Abaixo analisamos quais categorias do arXiv são mais prevalentes na amostra avaliada.


In [8]:
# Extração de todas as categorias ativas (desdobrando multi-label)
all_cats = []
for item in df['classes_reais_aapd'].dropna():
    cats = [c.strip() for c in str(item).split(',')]
    all_cats.extend(cats)

cat_counts = pd.Series(all_cats).value_counts().head(15).reset_index()
cat_counts.columns = ['Categoria arXiv', 'Frequência']

display(cat_counts)


,Categoria arXiv,Frequência
0,cs.IT,683
1,math.IT,683
2,cs.LG,294
3,cs.AI,211
4,stat.ML,185
5,cs.DS,161
6,cs.DM,137
7,cs.SI,134
8,cs.LO,132
9,math.CO,119


### Visualização 5: Top 15 Categorias Oficiais do arXiv Presentes na Amostra


In [9]:
fig_cats = px.bar(
    cat_counts,
    x='Frequência',
    y='Categoria arXiv',
    orientation='h',
    title='Top 15 Categorias do arXiv na Amostra do AAPD (2.000 Documentos)',
    text='Frequência',
    color='Frequência',
    color_continuous_scale='Blues'
)

fig_cats.update_layout(
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=520
)

fig_cats.show()


## 7. Inspeção Qualitativa Lado a Lado (Estudo de Casos)

Comparamos diretamente resumos reais, as categorias oficiais do arXiv, o tópico gerado pelo LLDA e as palavras-chave do LDA.


In [10]:
# Seleção de amostras diversificadas para inspeção lado a lado
amostras_idx = [51, 178, 228, 450, 1024]
amostras = df[df['doc_id'].isin(amostras_idx)][
    ['doc_id', 'classes_reais_aapd', 'topico_llda', 'confianca_llda', 'topico_lda_classico', 'probabilidade_lda_classico', 'resumo_texto']
].copy()

display(amostras)


,doc_id,classes_reais_aapd,topico_llda,confianca_llda,topico_lda_classico,probabilidade_lda_classico,resumo_texto
0,51,"math.CO, cs.DM",Graph Edge Rabbit Hunting Problem,0.333333,"graph, graphs, time, algorithm, log, problem",0.688232,"we investigate hunters rabbit game , where a set of hunters tries to catch an invisible rabbit that slides along the..."
1,178,"physics.soc-ph, cs.DL",Scientific Research Collaboration Dynamics,0.333333,"problem, algorithm, algorithms, optimization, power, method",0.676931,collaborations and citations within scientific research grow simultaneously and interact dynamically modelling the c...
3,228,"cs.IT, math.IT",Quantum code construction from cyclic codes,0.333333,"codes, code, decoding, complexity, linear, coding",0.405525,"let r f 2 u f 2 u 2 f 2 be a non chain finite commutative ring , where u 3 u in this paper , we mainly study the con..."
460,450,"cs.SE, cs.LO, cs.PL",Calculus of Object Programs,1.000000,"model, time, approach, network, based, results",0.908299,verifying properties of object oriented software requires a method for handling references in a simple and intuitive...
1031,1024,"cs.IT, math.IT",Information Theoretic Analysis of Multi Keyhole Channel,0.500000,"channel, network, relay, power, transmission, multiple",0.468735,"an information theoretic analysis of a multi keyhole channel , which includes a number of statistically independent ..."


## 8. Sumário Final e Conclusões

### Q&A
* **O que o experimento comparativo avaliou?**
  Avaliou o desempenho e comportamento da modelagem de tópicos generativa baseada em LLMs locais (**LLDA**) em contrapartida ao modelo probabilístico tradicional **LDA Clássico** ($K=54$) sobre uma amostra de 2.000 resumos acadêmicos do dataset multi-label AAPD (arXiv).
* **Quais foram as principais diferenças de diversidade lexical encontradas?**
  O **LLDA** obteve uma taxa de **Distinct-2 de 84,01%** e **Distinct-1 de 27,20%**, gerando **1.989 tópicos únicos** a partir de um vocabulário de **2.322 palavras distintas**. Em contraste, o **LDA Clássico** obteve **Distinct-2 de 2,63%** e **Distinct-1 de 1,42%**, ficando restrito aos seus **54 tópicos latentes fixos** formados por um vocabulário conjunto de apenas **170 palavras únicas**.

### Data Analysis Key Findings
* **Expressividade Semântica vs. Agrupamento Probabilístico:** O LLDA formula títulos conceituais humanos concisos (média de 4,27 palavras por tópico, cumprindo a restrição de 2 a 5 palavras), enquanto o LDA produz combinações de 6 palavras-chave mais prováveis da distribuição latente de termos.
* **Diversidade Superior de Bigramas:** A diversidade lexical de bigramas (Distinct-2) do LLDA é mais de 31 vezes superior à do LDA clássico (84,01% vs. 2,63%), evidenciando que o modelo LLM quase nunca repete formulações conceituais idênticas entre documentos distintos.
* **Comportamento das Métricas de Confiança:** A probabilidade a posteriori do LDA apresentou média de 63,9% (desvio padrão de 0,25, variando de 18,0% a 99,4%), refletindo a certeza da mistura temática latente. Já a confiança do LLDA obteve média de 54,5% (desvio padrão de 0,15, com 90,0% das amostras situadas entre 0,50 e 1,00), medindo a consistência de votos entre feixes estocásticos independentes gerados com temperatura 0.7.
* **Perfil do Corpus AAPD:** As categorias de Teoria da Informação (`cs.IT` e `math.IT`, com 683 documentos cada) são predominantes no corpus, seguidas por Aprendizado de Máquina (`cs.LG` com 294 e `stat.ML` com 185) e Inteligência Artificial (`cs.AI` com 211).

### Insights or Next Steps
* **Próximo Passo (Agregação Semântica com BERTopic):** Como o LLDA gera tópicos altamente específicos (1.989 únicos em 2.000 artigos), a etapa natural seguinte no pipeline é aplicar clusterização densa (ex: BERTopic com embeddings de Sentence-Transformers + HDBSCAN, conforme estruturado em `experimento_bertopic/bertopic_aggregator.py`) para consolidar esses micro-tópicos conceituais em macro-temas globais interpretáveis.
* **Avaliação de Coerência e Alinhamento:** Mensurar a coerência semântica e a concordância mútua (*purity / mutual information*) entre os agrupamentos do LLDA/BERTopic e as 54 classes oficiais do arXiv.
